In [ ]:
import json
from dataclasses import dataclass

In [8]:
@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int

def ride_deserializer(data):
    json_str = data.decode("utf-8")
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

In [9]:
test_bytes = json.dumps({
    "PULocationID": 79, 
    "DOLocationID": 125, 
    "trip_distance": 1.52, 
    "total_amount": 14.16, 
    "tpep_pickup_datetime": 1611397873000
}).encode("utf-8")

ride_deserializer(test_bytes)

Ride(PULocationID=79, DOLocationID=125, trip_distance=1.52, total_amount=14.16, tpep_pickup_datetime=1611397873000)

In [10]:
from kafka import KafkaConsumer

server = "localhost:9092"
topic_name = "rides"

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset="earliest",
    group_id="rides-console",
    value_deserializer=ride_deserializer,
)

/var/folders/jf/d7wy_nf16_gfmwtg3dp20v7r0000gn/T/ipykernel_69969/1295754031.py:6: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


In [11]:
from datetime import datetime

print(f"Listening to {topic_name}...")

count = 0

for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
          f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
          f"pickup={pickup_dt}")
    count += 1
    if count >= 10:
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break

consumer.close()

Listening to rides...
Received: PU=161, DO=262, distance=2.4, amount=$18.50, pickup=2021-01-15 19:36:03
Received: PU=161, DO=262, distance=2.4, amount=$18.50, pickup=2021-01-15 19:36:03
Received: PU=161, DO=262, distance=2.4, amount=$18.50, pickup=2021-01-15 19:36:03
Received: PU=238, DO=237, distance=2.25, amount=$16.62, pickup=2021-01-28 09:51:29
Received: PU=132, DO=203, distance=5.42, amount=$18.80, pickup=2021-01-15 19:12:34
Received: PU=233, DO=161, distance=1.01, amount=$13.56, pickup=2021-01-21 14:49:30
Received: PU=230, DO=229, distance=1.0, amount=$12.95, pickup=2021-01-22 18:55:01
Received: PU=181, DO=133, distance=2.48, amount=$22.00, pickup=2021-01-15 16:57:24
Received: PU=237, DO=260, distance=4.2, amount=$20.30, pickup=2021-01-22 13:33:44
Received: PU=162, DO=262, distance=2.21, amount=$15.96, pickup=2021-01-12 11:59:54

... received 10 messages so far (stopping after 10 for demo)
